In [0]:
-- =============================================================================
-- SILVER LAYER — Retail Sales Data Warehouse
-- Catalog : retail-dwh-project  |  Schema : silver
-- =============================================================================

USE CATALOG `retail-dwh-project`;



USE SCHEMA silver;





-- =============================================================================
-- TABLE 1 : DimCustomer  (SCD Type 2)
-- =============================================================================

CREATE TABLE IF NOT EXISTS `retail-dwh-project`.silver.DimCustomer (
    CustomerSK   BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID   INT,
    CustomerName STRING,
    Email        STRING,
    City         STRING,
    Address      STRING,
    StartDate    DATE,
    EndDate      DATE,
    IsActive     INT
)
USING DELTA
LOCATION 's3://retail-etl-lakehouse-gopika/processed/silver/dim_customer/';





-- STEP 2A: Expire old active records where City or Address changed
MERGE INTO `retail-dwh-project`.silver.DimCustomer AS target
USING (
    SELECT
        CustomerID,
        INITCAP(TRIM(CustomerName))  AS CustomerName,
        LOWER(TRIM(Email))           AS Email,
        TRIM(City)                   AS City,
        TRIM(Address)                AS Address
    FROM (
        SELECT
            CustomerID,
            CustomerName,
            Email,
            City,
            Address,
            LastUpdated,
            ROW_NUMBER() OVER (
                PARTITION BY CustomerID
                ORDER BY LastUpdated DESC
            ) AS rn
        FROM `retail-dwh-project`.bronze.customers_raw
        WHERE CustomerID IS NOT NULL
    )
    WHERE rn = 1
) AS source
ON  target.CustomerID = source.CustomerID
AND target.IsActive   = 1

WHEN MATCHED AND (
    target.City    <> source.City OR
    target.Address <> source.Address
)
THEN UPDATE SET
    target.EndDate  = CURRENT_DATE(),
    target.IsActive = 0

WHEN NOT MATCHED
THEN INSERT (
    CustomerID, CustomerName, Email, City, Address, StartDate, EndDate, IsActive
)
VALUES (
    source.CustomerID,
    source.CustomerName,
    source.Email,
    source.City,
    source.Address,
    CURRENT_DATE(),
    DATE('9999-12-31'),
    1
);





-- =============================================================================
-- TABLE 2 : DimProduct
-- =============================================================================

CREATE TABLE IF NOT EXISTS `retail-dwh-project`.silver.DimProduct (
    ProductSK     BIGINT,
    ProductID     INT,
    ProductName   STRING,
    Category      STRING,
    UnitPrice     DOUBLE,
    EffectiveDate DATE
)
USING DELTA
LOCATION 's3://retail-etl-lakehouse-gopika/processed/silver/dim_product/';





MERGE INTO `retail-dwh-project`.silver.DimProduct AS target
USING (
    SELECT
        ProductID,
        TRIM(ProductName)  AS ProductName,
        TRIM(Category)     AS Category,
        UnitPrice,
        CURRENT_DATE()     AS EffectiveDate
    FROM `retail-dwh-project`.bronze.products_raw
    WHERE ProductID IS NOT NULL
      AND UnitPrice > 0
) AS source
ON target.ProductID = source.ProductID

WHEN NOT MATCHED
THEN INSERT (
    ProductSK, ProductID, ProductName, Category, UnitPrice, EffectiveDate
)
VALUES (
    CAST(source.ProductID AS BIGINT),
    source.ProductID,
    source.ProductName,
    source.Category,
    source.UnitPrice,
    source.EffectiveDate
);





-- =============================================================================
-- TABLE 3 : DimStore
-- =============================================================================

CREATE TABLE IF NOT EXISTS `retail-dwh-project`.silver.DimStore (
    StoreSK   BIGINT,
    StoreID   INT,
    StoreName STRING,
    Region    STRING
)
USING DELTA
LOCATION 's3://retail-etl-lakehouse-gopika/processed/silver/dim_store/';





MERGE INTO `retail-dwh-project`.silver.DimStore AS target
USING (
    SELECT
        StoreID,
        TRIM(INITCAP(StoreName))          AS StoreName,
        COALESCE(TRIM(Region), 'Unknown') AS Region
    FROM `retail-dwh-project`.bronze.stores_raw
    WHERE StoreID IS NOT NULL
) AS source
ON target.StoreID = source.StoreID

WHEN NOT MATCHED
THEN INSERT (
    StoreSK, StoreID, StoreName, Region
)
VALUES (
    CAST(source.StoreID AS BIGINT),
    source.StoreID,
    source.StoreName,
    source.Region
);





-- =============================================================================
-- TABLE 4 : FactSales
-- =============================================================================

CREATE TABLE IF NOT EXISTS `retail-dwh-project`.silver.FactSales (
    SalesSK       BIGINT,
    TransactionID INT,
    CustomerSK    BIGINT,
    ProductSK     BIGINT,
    StoreSK       BIGINT,
    Quantity      INT,
    Amount        DECIMAL(10,2),
    TxnDate       DATE
)
USING DELTA
LOCATION 's3://retail-etl-lakehouse-gopika/processed/silver/fact_sales/';





MERGE INTO `retail-dwh-project`.silver.FactSales AS target
USING (
    SELECT
        ROW_NUMBER() OVER (ORDER BY s.TransactionID) AS SalesSK,
        s.TransactionID,
        c.CustomerSK,
        p.ProductSK,
        st.StoreSK,
        s.Quantity,
        ROUND(s.Quantity * p.UnitPrice, 2) AS Amount,
        s.TxnDate
    FROM (
        SELECT
            TransactionID,
            CustomerID,
            ProductID,
            StoreID,
            Quantity,
            TxnDate,
            ROW_NUMBER() OVER (
                PARTITION BY TransactionID
                ORDER BY TxnDate ASC
            ) AS rn
        FROM `retail-dwh-project`.bronze.sales_raw
        WHERE TransactionID IS NOT NULL
          AND Quantity > 0
    ) s
    INNER JOIN `retail-dwh-project`.silver.DimCustomer c
        ON s.CustomerID = c.CustomerID
       AND c.IsActive = 1
    LEFT JOIN `retail-dwh-project`.silver.DimProduct p
        ON s.ProductID = p.ProductID
    LEFT JOIN `retail-dwh-project`.silver.DimStore st
        ON s.StoreID = st.StoreID
    WHERE s.rn = 1
) AS source
ON target.TransactionID = source.TransactionID

WHEN NOT MATCHED
THEN INSERT (
    SalesSK, TransactionID, CustomerSK, ProductSK, StoreSK,
    Quantity, Amount, TxnDate
)
VALUES (
    source.SalesSK,
    source.TransactionID,
    source.CustomerSK,
    source.ProductSK,
    source.StoreSK,
    source.Quantity,
    source.Amount,
    source.TxnDate
);